# patch stithing error
projecting down the results fromm 3 to 1 dimension using PCA. Then correlating the result with the individual input variables.

1. load model
2. load dataset patches
3. predict results using NOFE and 6 dimensionality-reduction baselines (PCA, t-SNE, UMAP, PHATE, SpatialPCA, Concord)
4. stitch results together + plot
5. quantitative evaluation based on neighbors across patches

---
### Utilities

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.append(str(project_root / 'src'))
sys.path.append(str(project_root / 'ERA5' / 'scripts'))


# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from plot_utilities import *
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

from scipy.spatial import cKDTree

import phate

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config


# ---------------------------------------------------------------------------
# Shared prediction + stitching-error helpers.
#
# UMAP, Concord and SpatialPCA each only exist in their own conda env, so they are
# invoked as subprocesses. Buffer ids are per-process: with the fixed 'umap_buffer'
# names used by 2d_scatter.ipynb, two concurrently running notebooks race on the same
# .npy/.csv files.
METHOD_NAMES = ['nofe', 'pca', 'tsne', 'umap', 'phate', 'spatialpca', 'concord']

_spatialpca_proc = None
_spatialpca_dir = f'{project_root}/ERA5/spatialpca'
_spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'


def _spatialpca_server():
    """Start the persistent SpatialPCA R server once; return the process.

    Library load (SpatialPCA pulls in Seurat, SPARK, Rcpp) costs ~tens of seconds, so
    the server loads it once and then answers one request per stdin line.
    """
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        return _spatialpca_proc
    _spatialpca_proc = subprocess.Popen(
        ['conda', 'run', '-n', 'spatialpca', '--no-capture-output',
         'Rscript', 'execute_spatialpca_server.R'],
        cwd=_spatialpca_dir, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, text=True, bufsize=1,
        # R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the
        # conda env's own library. CRAN-built packages there link the CRAN R framework
        # rather than conda's libR, and SpatialPCA's Rcpp .so then segfaults at dyn.load.
        env=dict(os.environ, R_LIBS_USER='/nonexistent'),
    )
    ready = _spatialpca_proc.stdout.readline().strip()
    if ready != 'READY':
        raise RuntimeError(
            f'spatialPCA server failed to start: {ready!r}\n'
            f'{_spatialpca_proc.stderr.read()}'
        )
    return _spatialpca_proc


def spatialpca_embed(X, coords, n_components=3):
    """SpatialPCA embedding of X (samples x features) located at coords (samples x 3)."""
    proc = _spatialpca_server()
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv', coords, delimiter=',')

    proc.stdin.write(f'{_spatialpca_call_id} {n_components}\n')
    proc.stdin.flush()
    response = proc.stdout.readline().strip()

    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv')
    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {_spatialpca_call_id}: {response}')

    Y = np.loadtxt(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv',
                   delimiter=',', skiprows=1)
    os.remove(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv')
    return Y


def spatialpca_shutdown():
    """Stop the R server (call after the last SpatialPCA embedding)."""
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        _spatialpca_proc.stdin.write('QUIT\n')
        _spatialpca_proc.stdin.flush()
        _spatialpca_proc.wait(timeout=10)
    _spatialpca_proc = None


def external_embed(env, workdir, script, buffer_id, X):
    """Run `script` under conda env `env`: it reads <buffer_id>.npy, writes res_<buffer_id>.npy."""
    np.save(f'{workdir}/{buffer_id}.npy', X)
    subprocess.run(f'cd {workdir} && conda run -n {env} python {script} {buffer_id}', shell=True)
    Y = np.load(f'{workdir}/res_{buffer_id}.npy')
    os.remove(f'{workdir}/{buffer_id}.npy')
    os.remove(f'{workdir}/res_{buffer_id}.npy')
    return Y


def predict_all(graphs, model):
    """Predict 3-D embeddings for every patch, for every method in METHOD_NAMES.

    Returns {method: [array(n_nodes_patch_i, 3), ...]} in patch order.
    """
    pred_lists = {method: [] for method in METHOD_NAMES}

    for sample in graphs:
        print('NOFE ...', end='\r')
        pred_lists['nofe'].append(model(sample).detach().cpu().numpy())

        X = sample.normed_values.detach().cpu().numpy()

        print('PCA ...', end='\r')
        pred_lists['pca'].append(PCA(n_components=3).fit_transform(X))

        print('t-SNE ...', end='\r')
        pred_lists['tsne'].append(TSNE(
            n_components=3, perplexity=30, learning_rate='auto', random_state=42
        ).fit_transform(X))

        print('UMAP ...', end='\r')
        pred_lists['umap'].append(external_embed(
            'umap', f'{project_root}/ERA5/umap', 'execute_umap.py',
            f'umap_buffer_{os.getpid()}', X,
        ))

        print('PHATE ...', end='\r')
        pred_lists['phate'].append(phate.PHATE(
            n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
        ).fit_transform(X))

        print('spatialPCA ...', end='\r')
        pred_lists['spatialpca'].append(
            spatialpca_embed(X, sample.normed_coords.detach().cpu().numpy())
        )

        print('Concord ...', end='\r')
        pred_lists['concord'].append(external_embed(
            'concord', f'{project_root}/ERA5/concord', 'execute_concord.py',
            f'concord_buffer_{os.getpid()}', X,
        ))

    return pred_lists


def mean_dist(x):
    """Mean pairwise euclidean distance inside one embedding."""
    N = x.shape[0]
    dists = torch.cdist(x, x)
    return dists.triu(diagonal=1).sum() / (N * (N - 1) / 2)


def stitching_error(pred_lists, graphs, edge_idx_pairs, edge_subset_pairs):
    """Cross-patch disagreement of one embedding set, normalized per patch.

    For every border pair (a, b) both directions are measured: each border node of a is
    compared against its nearest coordinate neighbour among the border nodes of b, and
    vice versa. Each distance is divided by the mean pairwise distance of the patch it
    starts from, so methods with different output scale are comparable.

    Returns {method: 1-D tensor of per-node normalized distances}.
    """
    errors = {}
    for method in METHOD_NAMES:
        output_data = pred_lists[method]
        normed_distance_tensors = []

        for i in range(len(edge_idx_pairs)):
            a, b = edge_subset_pairs[i]
            a_idx, b_idx = edge_idx_pairs[i]

            coords_a = graphs[a].normed_coords[a_idx, :].detach().cpu().numpy()
            coords_b = graphs[b].normed_coords[b_idx, :].detach().cpu().numpy()

            # ============ first from patch A to B ============
            _, indices_a = cKDTree(coords_b).query(coords_a)
            A = torch.from_numpy(output_data[a][a_idx])
            B = torch.from_numpy(output_data[b][b_idx][indices_a])
            normed_distance_tensors.append(torch.norm(A - B, dim=1) / mean_dist(A))

            # ============ then from patch B to A ============
            _, indices_b = cKDTree(coords_a).query(coords_b)
            B = torch.from_numpy(output_data[b][b_idx])
            A = torch.from_numpy(output_data[a][a_idx][indices_b])
            normed_distance_tensors.append(torch.norm(B - A, dim=1) / mean_dist(B))

        errors[method] = torch.hstack(normed_distance_tensors)
    return errors


----
### load model + dataset 

In [ ]:
model, config = load_model(id='regular_papermodel2_2026-05-04_t12-01-43' , model_type='regular')

In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [ ]:
lat_range = (lat_max - lat_min)/2
lon_range = (lon_max - lon_min)/2
margin=5
'''
bbox1 = lon_min, lat_min+lat_range-margin, lon_max-lon_range+margin, lat_max
bbox2 = lon_min+lon_range-margin, lat_min+lat_range-margin, lon_max, lat_max
bbox3 = lon_min, lat_min, lon_max-lon_range+margin, lat_max-lat_range+margin
bbox4 = lon_min+lon_range-margin, lat_min, lon_max, lat_max-lat_range+margin

bboxes = [bbox1, bbox2, bbox3, bbox4]
'''
bbox1 = lon_min, lat_min, lon_max-lon_range+margin, lat_max
bbox2 = lon_min+lon_range-margin, lat_min, lon_max, lat_max
bboxes = [bbox1, bbox2]

In [ ]:
date = "2019-04-17"

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)

N=10000
print('generate data')

graphs=[]
locations_list=[]

for bbox in bboxes:
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))

---
## Get Predictions

In [ ]:
# all 7 methods, one entry per patch, keyed by METHOD_NAMES
pred_lists = predict_all(graphs, model)


# Plot

In [ ]:
locations = pd.DataFrame(np.vstack(locations_list), columns=['lat', 'lon'])

# one dataframe per method: lat/lon of every patch point + its 3 embedding channels
combined = {}
for method in METHOD_NAMES:
    pred_combined = pd.DataFrame(
        np.vstack(pred_lists[method]), columns=['x1', 'x2', 'x3']
    )
    combined[method] = pd.concat([locations, pred_combined], axis=1)


In [ ]:
figsize = (5, 5)
figs = {method: rgb(combined[method], figsize=figsize) for method in METHOD_NAMES}


In [ ]:
output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)

for method, fig in figs.items():
    fig.savefig(output_dir / f"gluing_{method}_{date}_N{N}.pdf", bbox_inches="tight")

# Quantitative Evaluation

In [ ]:
edge_margin = 2

# patches overlap in longitude, so the shared border is the lon band around bbox1[2]/bbox2[0]
idx12 = np.where(graphs[0].meta.gps[:, 1] > (bbox1[2] - edge_margin))[0]
idx21 = np.where(graphs[1].meta.gps[:, 1] < (bbox2[0] + edge_margin))[0]

edge_idx_pairs = [(idx12, idx21)]
edge_subset_pairs = [(0, 1)]


In [ ]:
# stitching error for every method on the single date loaded in the cells above
errors = stitching_error(pred_lists, graphs, edge_idx_pairs, edge_subset_pairs)
for method in METHOD_NAMES:
    print(f'{method}: {errors[method].mean()}')


In [ ]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

Generator = Data_Generator(data_path)
N = 1000
edge_margin = 2

per_date_means = {method: [] for method in METHOD_NAMES}

for date in dates:
    # ==================== Generate Data ====================
    graphs_d = []
    for bbox in bboxes:
        Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
        print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
        graph = Generator.sample_graph(
            N=N, k=k, perplexity=perplexity, sampling_method=sampling_method,
            gen_feature_mode=feature_mode,
        )
        graphs_d.append(graph.to(device))

    # ==================== Predict ====================
    pred_lists_d = predict_all(graphs_d, model)

    # ==================== Quantify ====================
    idx12 = np.where(graphs_d[0].meta.gps[:, 1] > (bbox1[2] - edge_margin))[0]
    idx21 = np.where(graphs_d[1].meta.gps[:, 1] < (bbox2[0] + edge_margin))[0]

    errors_d = stitching_error(pred_lists_d, graphs_d, [(idx12, idx21)], [(0, 1)])
    for method in METHOD_NAMES:
        per_date_means[method].append(errors_d[method].mean().item())

    print(f'{date}: ' + ', '.join(
        f'{method}={per_date_means[method][-1]:.4f}' for method in METHOD_NAMES
    ))

spatialpca_shutdown()

print(f'\nmean ± std over {len(dates)} dates')
for method in METHOD_NAMES:
    t = torch.tensor(per_date_means[method])
    print(f'{method}:\t{t.mean():.4f} ± {t.std():.4f}')
